# Chapter 2: Mathematical Foundations

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch02_mathematical_foundations.ipynb)


## What is in this notebook, and what to change in it

Three cells, and none of them loads anything. Every number is computed from
constants written into the cell above it, so the whole file runs in about a
second and each result can be checked by hand.

1. **The entropy of English letters.** A published frequency table for 26
   letters plus space, normalised, and its Shannon entropy: 4.087 bits, against
   the 4.755 a uniform 27-symbol alphabet would cost. The ratio, 0.859, is how
   much of the alphabet's capacity English actually uses.
2. **Entropy against temperature**, for a five-word vocabulary. Dividing the
   logits by a larger T flattens the distribution and raises its entropy towards
   the 2.32-bit maximum the cell prints, and that curve is what every sampling
   frame from chapter 7 onwards refers back to. This cell forces the Agg
   backend and writes its plot to a file rather than displaying it, so there is
   no figure stored here to look at.
3. **Perplexity under add-one smoothing**, on the corpus "the cat sat on the
   mat": cross-entropy 2.264 bits per word, perplexity 4.80, against a uniform
   baseline of 5.

**One thing to notice about the third cell.** Its comment mentions 4.76 for the
unsmoothed version, and nothing in the cell computes that number; it refers to a
hand computation in the chapter text. Only 4.80 is checkable by running this.
That gap between what a comment asserts and what the code demonstrates is worth
looking for everywhere, and chapter 3's notebook prints both, so the comparison
can be made there instead.

Three edits worth making. Put another language's letter frequencies into the
first cell and watch the entropy move. Widen the gap between the logits in the
second and see how much hotter the softmax has to run to recover the same
entropy. Then add a word to the third cell's test sentence that is not in the
vocabulary: the unsmoothed estimate gives it probability zero, and chapter 3's
third cell shows what that does to a perplexity.


> **This notebook was executed when it was built**, so the output under each
> cell is a real run's and you can read the file without running anything.
> Rebuild it with `python tools/build_notebook.py ch02`.


### 2.3.1 Entropy of a Language

![Figure 2.2 -- Entropy visualization comparing high-entropy and low-entropy distributions](../figures/fig-02-2.pdf)


In [1]:
import numpy as np

# English letter frequencies (26 letters + space, from published tables)
freqs = np.array([
    0.0651, 0.0124, 0.0217, 0.0349, 0.1041,  # a-e
    0.0197, 0.0158, 0.0492, 0.0558, 0.0009,  # f-j
    0.0050, 0.0331, 0.0202, 0.0564, 0.0596,  # k-o
    0.0137, 0.0008, 0.0497, 0.0515, 0.0729,  # p-t
    0.0225, 0.0082, 0.0171, 0.0014, 0.0145,  # u-y
    0.0007, 0.1820                             # z, space
])
freqs = freqs / freqs.sum()  # ensure normalization

# Shannon entropy in bits
H = -np.sum(freqs * np.log2(freqs))
H_max = np.log2(len(freqs))  # maximum entropy (uniform)
ratio = H / H_max

print(f"Entropy of English letters: {H:.3f} bits")
print(f"Maximum entropy (uniform):  {H_max:.3f} bits")
print(f"Efficiency ratio:           {ratio:.3f}")
# Output: H ~ 4.08 bits, H_max = 4.75 bits, ratio ~ 0.86


Entropy of English letters: 4.087 bits
Maximum entropy (uniform):  4.755 bits
Efficiency ratio:           0.859


### 2.3.1 Entropy of a Language


In [2]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Entropy vs. temperature for a 5-word vocabulary
logits = np.array([2.0, 1.0, 0.5, 0.1, 0.01])
temperatures = np.linspace(0.1, 5.0, 100)
entropies = []

for T in temperatures:
    scaled = logits / T
    scaled -= scaled.max()  # numerical stability
    probs = np.exp(scaled) / np.exp(scaled).sum()
    H = -np.sum(probs * np.log2(probs + 1e-12))
    entropies.append(H)

H_max = np.log2(len(logits))
plt.figure(figsize=(8, 5))
plt.plot(temperatures, entropies, 'b-', linewidth=2)
plt.axhline(y=H_max, color='r', linestyle='--', label=f'H_max = {H_max:.2f} bits')
plt.xlabel('Temperature T')
plt.ylabel('Entropy (bits)')
plt.title('Entropy vs. Temperature')
plt.legend()
plt.savefig('entropy_vs_temperature.png', dpi=150)
print(f"H_max = {H_max:.2f} bits. Plot saved.")


H_max = 2.32 bits. Plot saved.


### 2.3.4 Perplexity


In [3]:
import numpy as np

# Unigram model from corpus "the cat sat on the mat"
vocab = ['the', 'cat', 'sat', 'on', 'mat']
train_counts = np.array([2, 1, 1, 1, 1])  # raw counts
V = len(vocab)

# Add-1 (Laplace) smoothing. The hand computation above uses raw
# (unsmoothed) MLE probabilities and yields perplexity 4.76; the
# smoothed version below yields a slightly higher perplexity (4.80)
# because smoothing pulls probabilities toward the uniform baseline.
smoothed = (train_counts + 1) / (train_counts.sum() + V)

# Test sentence (identical to the training corpus in this toy example)
test_sentence = ['the', 'cat', 'sat', 'on', 'the', 'mat']
log_probs = []
for word in test_sentence:
    idx = vocab.index(word)
    lp = np.log2(smoothed[idx])
    log_probs.append(lp)
    print(f"  P({word}) = {smoothed[idx]:.4f}, log2 = {lp:.3f}")

cross_entropy = -np.mean(log_probs)
perplexity = 2 ** cross_entropy

print(f"\nCross-entropy: {cross_entropy:.3f} bits/word")
print(f"Perplexity:    {perplexity:.2f}")
print(f"Vocab size:    {V} (uniform baseline PP = {V})")


  P(the) = 0.2727, log2 = -1.874
  P(cat) = 0.1818, log2 = -2.459
  P(sat) = 0.1818, log2 = -2.459
  P(on) = 0.1818, log2 = -2.459
  P(the) = 0.2727, log2 = -1.874
  P(mat) = 0.1818, log2 = -2.459

Cross-entropy: 2.264 bits/word
Perplexity:    4.80
Vocab size:    5 (uniform baseline PP = 5)


---

## Summary

This notebook demonstrated the key code examples from Chapter 2: Mathematical Foundations. For the full mathematical exposition and discussion, refer to the textbook chapter.
